# 04.03 · Build a tiny GPT — Milestone M1

**Chapter:** Transformers & How LLMs Work · **Milestone:** M1

## Goals
- [ ] Assemble a decoder-only transformer: embeddings → N × (attention + MLP blocks) → LayerNorm → vocab logits
- [ ] Train it on a small corpus with next-token prediction
- [ ] Generate text by sampling autoregressively

## Resources
- [Karpathy – Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY) — free (this notebook follows it)
- [karpathy/nanoGPT](https://github.com/karpathy/nanoGPT) — free
- [Raschka – LLMs from Scratch](https://github.com/rasbt/LLMs-from-scratch) — code free, book paid

Same architecture family as GPT-2/3, Llama and Qwen. The differences at scale are size, data, RoPE/RMSNorm/SwiGLU tweaks,
and post-training — not the core idea.

In [ ]:
import math
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
from torch import nn

from aieng import DATA

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(1337)
print(device)

## 1. Data: tiny Shakespeare, character-level
Character tokens keep the tokenizer trivial so we can focus on the model. (Exercise: switch to your own text.)

In [ ]:
path = DATA / "tinyshakespeare.txt"
if not path.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", path)
text = path.read_text()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = dict(enumerate(chars))
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"{len(text):,} characters, vocab {vocab_size}\n---\n{text[:200]}")

## 2. Batches
Sample random windows of `block_size` characters. Target = input shifted by one: at every position, predict the next char.
One window of length T gives T training examples at once (thanks to the causal mask).

In [ ]:
block_size, batch_size = 128, 32

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print(repr(decode(xb[0, :20].tolist())), "->", repr(decode(yb[0, :20].tolist())))

## 3. The model
A **block** is pre-norm attention + pre-norm MLP, each wrapped in a residual connection (03.05):
```
x = x + Attention(LayerNorm(x))
x = x + MLP(LayerNorm(x))
```

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.h = n_heads
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.drop = nn.Dropout(dropout)  # dropout on the output (SDPA's built-in dropout isn't supported on MPS)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))
        y = nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.drop(self.proj(y.transpose(1, 2).reshape(B, T, C)))


class Block(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = CausalSelfAttention(d, n_heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, d=192, n_layers=4, n_heads=6, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Embedding(block_size, d)  # learned absolute positions (GPT-2 style)
        self.blocks = nn.Sequential(*[Block(d, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying: input and output embeddings share parameters

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        logits = self.head(self.ln_f(self.blocks(x)))  # (B, T, vocab)
        loss = None if targets is None else nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size :])  # crop to context window
            probs = (logits[:, -1] / temperature).softmax(-1)  # only the last position matters
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


model = TinyGPT(vocab_size, block_size).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

Before training the loss should be ≈ $\ln(\text{vocab})$ — a uniform guess (01.04). If it isn't, something's wrong.

In [ ]:
_, loss = model(xb, yb)
print(f"initial loss {loss.item():.3f} vs ln(vocab) = {math.log(vocab_size):.3f}")
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 100)[0].tolist()))

## 4. Train
AdamW + warmup + cosine decay + gradient clipping: the standard LLM recipe in miniature. ~2–4 min on Apple Silicon / a Colab GPU.

In [ ]:
MAX_STEPS, WARMUP, LR = 2000, 100, 3e-3
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
lr_at = lambda s: LR * s / WARMUP if s < WARMUP else LR * 0.5 * (1 + math.cos(math.pi * (s - WARMUP) / (MAX_STEPS - WARMUP)))


@torch.no_grad()
def estimate_loss(iters=20):
    model.eval()
    out = {s: torch.stack([model(*get_batch(s))[1] for _ in range(iters)]).mean().item() for s in ("train", "val")}
    model.train()
    return out


history, start = [], time.perf_counter()
for step in range(MAX_STEPS + 1):
    for g in opt.param_groups: g["lr"] = lr_at(step)
    _, loss = model(*get_batch("train"))
    opt.zero_grad(set_to_none=True); loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 250 == 0:
        losses = estimate_loss(); history.append((step, losses["train"], losses["val"]))
        print(f"step {step:5}: train {losses['train']:.3f} | val {losses['val']:.3f} | {time.perf_counter() - start:.0f}s")

steps, tr, va = zip(*history)
plt.plot(steps, tr, label="train"); plt.plot(steps, va, label="val"); plt.legend(); plt.ylabel("cross-entropy"); plt.show()

## 5. Generate

In [ ]:
model.eval()
prompt = torch.tensor([encode("ROMEO:")], device=device)
for T in [0.5, 1.0]:
    print(f"--- temperature {T}\n{decode(model.generate(prompt, 300, temperature=T)[0].tolist())}\n")

torch.save({"model": model.state_dict(), "chars": chars, "block_size": block_size}, DATA / "tinygpt.pt")

A val loss around 1.5–1.7 gives recognisable Shakespeare-shaped text: names, line breaks, archaic words, nonsense meaning.
That gap — form without understanding — closes with scale: more parameters, more data, better tokens.

## Exercises — Milestone M1
1. **Train on your own text** (notes, chats export, a book you love). Generate 500 characters. Commit the samples to your repo.
2. Scale up: `d=384, n_layers=6, n_heads=6, block_size=256`, more steps (Colab GPU). How low can val loss go before overfitting?
3. Replace the character tokenizer with your BPE from 04.01 (or `tiktoken`). What changes in loss values and sample quality? Why aren't the losses directly comparable?
4. Replace LayerNorm with RMSNorm and GELU-MLP with SwiGLU (Llama-style). Any difference?

In [ ]:
# your code here